# Excitation Survival + Temperature Analysis

Interactive analysis of `ExcitationSurvivalScan` runs. Reuses the BLAS-free helpers from
`analyze_excitation_survival.py`:

- **Panel A** — survival vs total excitation cycles at release=0 (Wilson 68% CIs) → the
  `max_excitation_cycles` ceiling (how many cycles the atom survives).
- **Panel B** — recapture survival vs release time, one curve per cycle count → the atom
  **temperature proxy** (a longer release before recapture drops = colder atom).

Run with the **working-BLAS `python`** kernel (numpy 2.x / h5py / mpl 3.x); matplotlib
crashes in the artiq7_claude env.

In [ ]:
import sys, os, glob
from pathlib import Path
_ANALYSIS_DIR = r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis"
if _ANALYSIS_DIR not in sys.path:
    sys.path.insert(0, _ANALYSIS_DIR)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2
from analyze_excitation_survival import (
    RESULTS_DIR, SURVIVAL_TARGET, find_run, load_run, survival_table, wilson, _half_release,
)
print("Results dir       :", RESULTS_DIR)
print("Survival target   :", SURVIVAL_TARGET, "(the ceiling threshold)")

## 1. List recent runs

In [ ]:
files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*-ExcitationSurvivalScan.h5"), recursive=True),
               key=lambda f: int(os.path.basename(f).split("-")[0]))
if not files:
    print("No *-ExcitationSurvivalScan.h5 found yet.")
for f in files[-12:]:
    rid = int(os.path.basename(f).split("-")[0])
    d, s, _ = load_run(f)
    n = len(np.atleast_1d(d.get("esv_total_cycles", [])))
    nl = int(np.sum(np.atleast_1d(d.get("esv_atom_loaded", [0])).astype(int)))
    print(f"RID {rid:>5} | {n:>6} pts | {nl:>5} loaded | "
          f"cpg={s.get('cycles_per_group','?')} | recool={s.get('recool_mode','?')}")

## 2. Select a run

Set `RID` to an int, or `None` for the latest.

In [ ]:
RID = None   # <-- int or None (latest)

path = find_run(RID)
data, snap, rid = load_run(path)
table = survival_table(data)                  # filters loaded==1
cycle_counts = sorted({c for (c, r) in table})
releases = sorted({r for (c, r) in table})

print("File :", os.path.basename(path))
print(f"RID {rid} | cycles_per_group={snap.get('cycles_per_group')} | recool={snap.get('recool_mode')}")
print("cycle counts      :", cycle_counts)
print("release times (us):", releases)

## 3. Panel A — survival vs cycles (the `max_excitation_cycles` ceiling)

In [ ]:
rel0 = 0.0 if 0.0 in releases else (releases[0] if releases else 0.0)
xs, ps, los, his = [], [], [], []
ceiling = None
for c in cycle_counts:
    kn = table.get((c, rel0))
    if not kn or kn[1] == 0:
        continue
    p, lo, hi = wilson(kn[0], kn[1])
    xs.append(c); ps.append(p)
    # clamp to >= 0: Wilson is centred on its own centre (not p), so at p=0/1
    # the error-bar distance can come out as a tiny negative from FP rounding,
    # which matplotlib's errorbar rejects.
    los.append(max(0.0, p - lo)); his.append(max(0.0, hi - p))
    if p >= SURVIVAL_TARGET:
        ceiling = c

fig, ax = plt.subplots(figsize=(7, 4.5))
if xs:
    ax.errorbar(xs, ps, yerr=[los, his], fmt="o-", capsize=4, color="C0")
    ax.axhline(SURVIVAL_TARGET, ls="--", color="gray", lw=1, label=f"target {SURVIVAL_TARGET:.0%}")
    if ceiling is not None:
        ax.axvline(ceiling, ls=":", color="C3", lw=1.5, label=f"ceiling ≈ {ceiling} cyc")
    ax.legend()
ax.set_xlabel("total excitation cycles")
ax.set_ylabel(f"survival (release={rel0:g} us)")
ax.set_ylim(0, 1.05)
ax.set_title(f"Survival vs cycles — RID {rid}")
plt.show()

cpg = int(snap.get("cycles_per_group", 4))
if ceiling is not None:
    print(f"Max cycles with survival >= {SURVIVAL_TARGET:.0%}: {ceiling}  "
          f"(= {ceiling // cpg} groups of {cpg})")
else:
    print("Survival never reached the target at release=0 -- lower SURVIVAL_TARGET or check recool/loading.")

## 4. Panel B — recapture vs release (temperature proxy)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for c in cycle_counts:
    rx, ry = [], []
    for r in releases:
        kn = table.get((c, r))
        if kn and kn[1] > 0:
            rx.append(r); ry.append(kn[0] / kn[1])
    if len(rx) >= 2:
        ax.plot(rx, ry, "o-", label=f"{c} cyc", lw=1.5, ms=4)
        half = _half_release(rx, ry)
        print(f"cycles={c:>5}: half-recapture release = " + (f"{half:.1f} us" if half is not None else "n/a"))
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("release time (us)")
ax.set_ylabel("recapture survival")
ax.set_ylim(0, 1.05)
ax.set_title("Recapture vs release (colder = decays later)")
ax.legend(fontsize=8, ncol=2)
plt.show()

## 5. (optional) one-shot PNG via the script

`analyze(path)` writes a 2-panel PNG next to the `.h5`.

In [ ]:
# from analyze_excitation_survival import analyze
# analyze(path)